# 两层 GCN —— 纯标准库 + 稀疏矩阵

参考 [Planetoid 数据集详解](https://blog.csdn.net/PolarisRisingWar/article/details/118714542) 里
`ind.*.x`（scipy CSR 稀疏特征）与 `ind.*.graph`（邻接字典）的数据形态，
把稠密矩阵乘法换成**稀疏行存 + 稀疏乘核**，让 Cora / CiteSeer / PubMed 三个数据集都能在
**只用 Python 标准库**的条件下跑完。

## 为什么必须稀疏化

| 数据集 | 节点 N | 特征维度 C | 类别 F | X中非零节点个数 nnz(X) | （number no zero）nnz(A) |
|---|---|---|---|---|---|
| Cora | 2708 | 1433 | 7 | 49 216 | 13 264 |
| CiteSeer | 3327 | 3703 | 6 | 105 165 | 14 542 |
| PubMed | 19717 | 500 | 3 | 988 031 | 108 365 |

- PubMed 的**稠密**邻接矩阵是 19717×19717 ≈ **3.9 亿**个 float，A大约需要16GB内存才存得下；
- 连稠密特征矩阵也有 19717×500 ≈ **985 万**个 float，W_0大约需要10MB内存；
- 稠密乘法的复杂度是 `O(N²C)`，而稀疏是 `O(nnz·C)`，差好几十倍。

所以特征和邻接**都必须**用稀疏表示。

## 变量名

```text
X        (N, C)  全部节点特征，C 是每个节点的特征数（稀疏行存）
X[i]     (cols, vals)  第 i 个节点的非零特征下标与取值
A        (N, N)  邻接矩阵（稀疏行存）
A_hat    (N, N)  归一化邻接矩阵（稀疏行存）
W_0      (C, H)  输入到隐藏层的权重
W_1      (H, F)  隐藏层到输出的权重
H_1      (N, H)  隐藏层激活
Z        (N, F)  行方向 softmax 后的类别概率
Y        (N, F)  one-hot 标签
N, C, H, F       节点数、输入维度、隐藏维度、类别数
```

## 公式（Kipf & Welling 2016）

预处理：

```text
A_tilde = A + I
D_tilde = diag(A_tilde 的行和)
A_hat   = D_tilde^(-1/2) · A_tilde · D_tilde^(-1/2)
```

前向：

```text
Z = softmax( A_hat · ReLU( A_hat · X · W_0 ) · W_1 )
```

损失：

```text
L_0 = -Σ_{i∈train} Σ_f  Y[i,f] · ln Z[i,f]
L   = L_0 + lambda_reg · (||W_0||² + ||W_1||²) / 2
```

## 三处关键优化（相对稠密版）

1. **稀疏行存 + 稀疏乘核**：只有非零元参与乘法，`O(nnz·C)` 代替 `O(N²C)`。
2. **2 跳局部化**：`Z_i` 只依赖 2 跳内的 `X`（`Z ← Â·H₁ ← H₁ ← Â·(X·W₀)`），
   所以算 train/val/test 节点的 `Z`，只需算各自 2 跳闭包内的行。
   PubMed 上训练侧闭包只有 2798/19717 ≈ 14% 的节点。
3. **逐节点确定性随机**：dropout 掩码由 `(seed, epoch, node, salt)` 决定，
   与「算哪些行」无关 —— 这保证了局部化版本与全图版本**逐位一致**（第 9 节验证）。

## 0 模块导入

只 import 标准库：`csv / json / math / os / random / time` + `operator` + `pathlib`。

In [ ]:
import csv
import json
import math
import os
import random
import time
from operator import itemgetter, mul
from pathlib import Path

CWD = Path.cwd()
HERE = CWD.parent if CWD.name == "notebooks" else CWD
DATA_ROOT = HERE / "data" / "csv"

## 1 稀疏矩阵内核

### 1.1 稀疏行存

行存就是**只存每行的非零元**：

```python
X[i] = (cols, vals)   # 第 i 行：cols 是非零元的列下标，vals 是对应取值
```

这正是参考文章里 `ind.*.x` 那个 `scipy.sparse.csr_matrix` 的 `indices` / `data`，
只不过这里用纯 Python 的 list 表达。

### 1.2 稀疏 × 稠密

对 `S`（行存稀疏，`m×k`）和稠密 `B`（`k×p`），只算 `S` 的某些行：

```text
out[i][h] = Σ_f  vals[i][f] · B[cols[i][f]][h]
```

`B` 先转置成列（`bt[h]` 是第 h 列），于是每行只要一次 `itemgetter` 取值 +
一次 `sum(map(mul, ...))` 乘加 —— **乘加全部在 C 层完成**，
Python 层每个输出元素只跑一次循环（比逐元素 Python 循环快约 30%）。

### 1.3 稀疏 × 稀疏映射

`Â·(X·W₀)` 和 `Â·H₁` 的右边（`X·W₀`、`H₁`）只算了一部分行，
所以用 **dict（行号 → 行向量）** 来存，右边就按行号查表。

### 1.4 转置乘

`Xᵀ·dU` 里 `Xᵀ` 的行就是 `X` 的列，所以按 `X` 的行扫描、
向 `dU` 的对应**列**累加（与 `X·dU` 的「沿行做行和」方向正好相反）。

### 1.5 逐节点确定性随机

`xorshift32`，种子 `(seed, epoch, node, salt)`。

> **为什么不用全局 `random`？**
> 局部化之后「这一轮算了哪些行」会变，如果 dropout 掩码来自一个共享的
> 全局随机流，同一节点的掩码就会随计算顺序漂移，局部化与全图的结果不再一致。
> 每个节点一条独立流 ⇒ 掩码只取决于**节点身份**，与算多少行、什么顺序都无关。

In [ ]:
def spmm_sparse_dense(S, B, rows, p)->{}:
    """行存稀疏 S(m×k) · 稠密 B(k×p)，只算 rows 指定的行。
    S[i] = (cols, vals)，第 i 行的非零元。复杂度 O(nnz(S)·p)，与 m 无关。
    内层展开：out[i][h] = Σ_f vals[i][f] · B[cols[i][f]][h]
    把 B 转置成列后，用 itemgetter 一次取出该行需要的 k 个值，
    再用 sum(map(mul, ...)) 做乘加 —— 乘加在 C 层完成。
    """
    bt = list(zip(*B))                       # bt[h] 是 B 的第 h 列
    out = {}
    for i in rows:
        cols, vals = S[i]
        k = len(cols)
        if k == 0:
            out[i] = [0.0] * p               # 整行都是零
        elif k == 1:                         # 在这里，实现了向量的快速点积
            c = cols[0]
            v = vals[0]
            out[i] = [v * col[c] for col in bt]
        else:
            g = itemgetter(*cols)            # 一次取 k 个值（C 层）
            out[i] = [sum(map(mul, vals, g(col))) for col in bt]
    return out


def spmm_sparse_map(S, D, rows, p):
    """行存稀疏 S · 稠密向量字典 D（行号 → 行向量），只算 rows 指定的行。

    用于 Â·(X·W₀) 与 Â·H₁：右边只算了一部分行，用 dict 按行号查表。
    """
    out = {}
    for i in rows:
        cols, vals = S[i]
        acc = [0.0] * p
        for j, a in zip(cols, vals):
            b = D[j]
            for h in range(p):
                acc[h] += a * b[h]
        out[i] = acc
    return out


def spmm_sparse_T(S, D, rows, p, n_cols):
    """Sᵀ·D：S 是行存稀疏 (m×k)，D 是 m×p 稠密（dict：行号 → 行向量）。

    返回 (k×p) 稠密矩阵。Sᵀ 的行就是 S 的列，
    所以按 S 的行扫描、向 D 的对应**列**累加（与 S·D 方向相反）。
    """
    out = [[0.0] * p for _ in range(n_cols)]
    for i in rows:
        cols, vals = S[i]
        d = D[i]
        for j, v in zip(cols, vals):
            o = out[j]
            for h in range(p):
                o[h] += v * d[h]
    return out


def dmm(A, B):
    """稠密 (m×k)·(k×p)。"""
    cols = list(zip(*B))
    return [[sum(map(mul, row, col)) for col in cols] for row in A]


class NodeRNG:
    """xorshift32：每个节点一条独立随机流，种子 (seed, epoch, node, salt)。

    掩码只取决于节点身份，与「这一轮算了哪些行」无关，
    这是局部化结果与全图结果逐位一致的前提。
    """

    __slots__ = ("s",)

    def __init__(self, seed, epoch, node, salt=0):
        s = (seed * 0x9E3779B1 + epoch * 0x85EBCA6B + node * 0xC2B2AE35 + salt) & 0xFFFFFFFF
        self.s = s or 0x12345678

    def random(self):
        s = self.s
        s ^= (s << 13) & 0xFFFFFFFF
        s ^= s >> 17
        s ^= (s << 5) & 0xFFFFFFFF
        self.s = s
        return s / 4294967296.0

## 2 数据

### 2.1 加载

对照参考文章里的数据形态：

| Planetoid 原始文件 | 本仓库对应 | 结构 |
|---|---|---|
| `ind.*.x` | `features.csv` | 稀疏长表 `node_id,feature_id,value` → 转成行存 |
| `ind.*.graph` | `edges.csv` | 邻接表 `source,target,weight` → 转成邻接集合 |
| `ind.*.y` / `tx.*` | `nodes.csv` | `node_id,split,label` |

`features.csv` 是**稀疏长表**（每行一个非零元），比 `features_dense.csv`
省得多 —— PubMed 的稠密宽表是 27 MB 文本、985 万个 float。

`edges.csv` 已经双向存储过，这里统一**去重 + 去掉自环**，
自环留到 `Â = D̃^(-1/2)(A+I)D̃^(-1/2)` 时统一加。

### 2.2 归一化邻接矩阵

`Â` 保持稀疏行存。`Â` 是**对称**矩阵（`D̃^(-1/2)` 两边各一个），
所以 `Âᵀ·v` 与 `Â·v` 结果相同 —— 反向传播里的转置乘可以直接复用同一套行结构。

### 2.3 2 跳闭包

`closure(adj, seeds, k)` 返回 `seeds` 的 k 跳闭包（含 seeds 自身）。

In [ ]:
def load_dataset(name):
    """读 CSV 版数据集，返回稀疏特征、邻接表、one-hot 标签与三个 mask。

    返回：X, adj, Y, train_mask, val_mask, test_mask, C, F
      X[i] = (cols, vals)  —— 第 i 节点的非零特征（行存）
      adj[i] = {邻居节点号}
    """
    data_dir = DATA_ROOT / name
    with open(data_dir / "meta.json", encoding="utf-8") as f:
        meta = json.load(f)
    num_classes = meta["num_classes"]

    # ---- 节点：标签 + 划分 ----
    labels, splits = [], []
    with open(data_dir / "nodes.csv", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)                                # 跳过表头
        for _node_id, split, label in reader:
            labels.append(int(label))
            splits.append(split)

    num_nodes = len(labels)

    # one-hot：Y[i, labels[i]] = 1；无标签节点（label = -1）整行为 0
    Y = [[0.0] * num_classes for _ in range(num_nodes)]
    for i, label in enumerate(labels):
        if label >= 0:
            Y[i][label] = 1.0
    train_mask = [s == "train" for s in splits]
    val_mask = [s == "val" for s in splits]
    test_mask = [s == "test" for s in splits]

    # ---- 特征：稀疏长表 → 行存 ----
    # features.csv 每行是 (node_id, feature_id, value)，直接按 node_id 归桶
    raw = [[] for _ in range(num_nodes)]
    with open(data_dir / "features.csv", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for node_id, feature_id, value in reader:
            raw[int(node_id)].append((int(feature_id), float(value)))
    X = [([j for j, _ in row], [v for _, v in row]) for row in raw]

    # ---- 邻接：邻接表 ----
    # edges.csv 已双向存储；这里去重、剔自环（自环稍后由 A+I 统一加入）
    adj = [set() for _ in range(num_nodes)]
    with open(data_dir / "edges.csv", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for u, v, _weight in reader:
            u, v = int(u), int(v)
            if u != v:
                adj[u].add(v)
                adj[v].add(u)

    return X, adj, Y, train_mask, val_mask, test_mask, meta["num_features"], num_classes


def row_normalize(X):
    """特征按行归一化（论文 §5.1：「row-normalize input feature vectors」）。

    稀疏行只缩放非零元，代价 O(nnz(X)) 而不是 O(N·C)。
    """
    for i, (cols, vals) in enumerate(X):
        total = sum(vals)
        if total > 0.0:               # 全零行保持全零，不做 0/0
            X[i] = (cols, [v / total for v in vals])
    return X


def build_A_hat(adj, num_nodes):
    """A_hat = D_tilde^(-1/2) (A + I) D_tilde^(-1/2)，保持稀疏行存。

    A_hat[i] = (nbrs, vals)，nbrs 含所有邻居 + 自己，vals = 1/sqrt(d_i·d_j)。

    A_hat 是**对称**矩阵，所以 A_hatᵀ·v ≡ A_hat·v ——
    反向传播里的转置乘可以直接复用同一套行结构，不需要第二套内核。
    """
    deg = [len(a) for a in adj]
    d = [math.sqrt(x + 1.0) for x in deg]        # D̃ = deg + 1（自环已计入）
    A = []
    for i in range(num_nodes):
        nbrs = list(adj[i])
        vals = [1.0 / (d[i] * d[j]) for j in nbrs]
        nbrs.append(i)                           # + I
        vals.append(1.0 / (d[i] * d[i]))
        A.append((nbrs, vals))
    return A


def closure(adj, seeds, k):
    """seeds 的 k 跳闭包（含 seeds 自身）。

    闭包必须满足下面两条，否则稀疏乘会去取没算过的行：
        S1 = closure(Sout, 1) ⊇ Sout ∪ N(Sout)   （H₁ 的需求行）
        S2 = closure(Sout, 2) ⊇ S1 ∪ N(S1)       （X·W₀ 的需求行）
    """
    cur = set(seeds)
    for _ in range(k):
        nxt = set(cur)
        for i in cur:
            nxt |= adj[i]
        cur = nxt
    return cur

## 3 初始化

> Glorot 均匀初始化，界是 `√(6/(fan_in+fan_out))`。
> 目标：让每一层输入、输出的方差尽量稳定，避免梯度爆炸 / 消失。

In [ ]:
def glorot(rows, cols, rng):
    """Glorot 均匀初始化：U(-sqrt(6/(fan_in+fan_out)), +...)。

    - `rows`：权重矩阵**输出维度（fan_out）**
    - `cols`：权重矩阵**输入维度（fan_in）**
    - `bound = sqrt(6/(fan_in + fan_out))`：均匀分布上下界
    - 返回 `rows × cols` 的二维列表，每个值从 `[-bound, bound]` 均匀采样
    """
    bound = math.sqrt(6.0 / (rows + cols))
    return [[rng.uniform(-bound, bound) for _ in range(cols)] for _ in range(rows)]

## 4 前向 / 反向

> dropout 加在**两层的输入端**（论文 §5.1：dropout rate for all layers）：
> `Â · dropout(X) · W_0` 与 `Â · dropout(H_1) · W_1`，两处共用同一个 keep_prob。
> `Â·X·W_0` 按结合律先算 `X·W_0` 再乘 `Â`——H 远小于 C（16 vs 1433），
> 每轮是 `O(nnz(X)·H + nnz(A)·H)` 而不是 `O(N²·C)`。
> 反向里 ReLU 的导数由 `pre_1 > 0` 给出。

### 4.1 局部化：三个 split 各自算各自的闭包

`Z_i` 只依赖 2 跳内的 `X`，所以：

- 训练轮：`X` 只需 `closure(train, 2)`，`H₁` 只需 `closure(train, 1)`；
- 验证轮：只需 `closure(val, 2)` / `closure(val, 1)`；
- 测试轮：只需 `closure(test, 2)` / `closure(test, 1)`。

三者**互不共用**。PubMed 上验证集的 2 跳闭包有 10964 个节点，
如果三个 split 共用全集（16419），每轮验证都要多算 1/3 的 `X·W₀`。

### 4.2 稀疏 dropout 与稠密 dropout 等价

`dropout` 对**零元无影响**（`0 · mask` 还是 `0`），所以第 1 层的 dropout
只作用在 `X` 的**非零元**上，与稠密版数学完全等价，
代价从 `O(N·C)` 降到 `O(nnz(X))`。

第 2 层的 dropout 作用在 `H₁` 上（本来就是稠密的），用 `NodeRNG` 逐节点抽掩码。

In [ ]:
def dropout_sparse(X, rows, keep_prob, seed, epoch, training):
    """第 1 层输入端的 dropout，作用在稀疏特征的非零元上。

    稠密 dropout 对零元无影响（0·mask 还是 0），
    所以只在非零元上抽掩码与稠密版完全等价，
    但代价从 O(N·C) 降到 O(nnz(X))。
    """
    if not training or keep_prob >= 1.0:
        return {i: X[i] for i in rows}
    out = {}
    for i in rows:
        cols, vals = X[i]
        rng = NodeRNG(seed, epoch, i, 0)          # salt=0：第 1 层
        out[i] = (cols, [v / keep_prob if rng.random() < keep_prob else 0.0
                         for v in vals])
    return out


def dropout_dense(H1, keep_prob, seed, epoch, training):
    """第 2 层输入端的 dropout，作用在 H₁ 的每个元素上。

    返回 (H1_in, mask)：
      H1_in —— 送进第 2 层的值，被丢弃的位置为 0
      mask  —— 已含 `1/keep` 还原的掩码，反向时用它把梯度乘回去
    """
    if not training or keep_prob >= 1.0:
        return ({i: H1[i] for i in H1},
                {i: [1.0] * len(H1[i]) for i in H1})
    out, mask = {}, {}
    for i, row in H1.items():
        rng = NodeRNG(seed, epoch, i, 1)          # salt=1：第 2 层
        m = [1.0 if rng.random() < keep_prob else 0.0 for _ in row]
        mask[i] = [mk / keep_prob for mk in m]
        out[i] = [(v * mk) / keep_prob for v, mk in zip(row, m)]
    return out, mask


def forward(G, W_0, W_1, keep_prob, seed, epoch, training, S2, S1, S_out):
    """两层前向。

    S2 = 需要 X 的行，S1 = 需要 H₁ 的行，S_out = 需要 Z 的行。
    两层都在**输入端**做 dropout（论文 §5.1：dropout rate for all layers）：

        H_1 = ReLU( Â · dropout(X) · W_0 )
        Z   = softmax( Â · dropout(H_1) · W_1 )

    返回 Z 与反向需要的中间量。
    """
    H = len(W_0[0])
    X_in = dropout_sparse(G["X"], S2, keep_prob, seed, epoch, training)
    XW = spmm_sparse_dense(X_in, W_0, S2, H)          # X·W₀
    pre_1 = spmm_sparse_map(G["A"], XW, S1, H)         # Â·(X·W₀)
    H_1 = {i: [v if v > 0.0 else 0.0 for v in row] for i, row in pre_1.items()}
    H_1_in, mask_1 = dropout_dense(H_1, keep_prob, seed, epoch, training)
    A_H = spmm_sparse_map(G["A"], H_1_in, S_out, H)    # Â·H₁，反向算 dW₁ 还要用
    Z = dict(zip(S_out, softmax_rows(dmm([A_H[i] for i in S_out], W_1))))
    return Z, (X_in, pre_1, A_H, mask_1)


def softmax_rows(rows):
    """按行 softmax，先减去行最大值以免 exp 溢出。"""
    out = []
    for row in rows:
        top = max(row)
        exp_row = [math.exp(v - top) for v in row]
        total = sum(exp_row)
        out.append([v / total for v in exp_row])
    return out


def backward(G, Z, cache, train_rows, W_1, lambda_reg):
    """交叉熵 + L2 的梯度，返回 (d_W_0, d_W_1) 与 L_0。"""
    X_in, pre_1, A_H, mask_1 = cache
    C = len(G["W_0"][0]) and len(G["W_0"])       # 特征维度
    F = len(W_1[0])
    num_train = len(train_rows)

    # 只在训练节点上算交叉熵（论文 eq.3 的 Y_L），取平均而不是求和，
    # 这样学习率不随训练集大小变化。softmax + 交叉熵对 logits 的梯度
    # 就是 (Z - Y) / N_train，一起在这里算掉。
    L_0 = 0.0
    d_logits = {}
    for i in train_rows:
        z = Z[i]
        y = G["Y"][i]
        dl = []
        for f in range(F):
            L_0 -= y[f] * math.log(z[f] + 1e-12)
            dl.append((z[f] - y[f]) / num_train)
        d_logits[i] = dl
    L_0 /= num_train

    # ---- W_1 的梯度 ----
    # logits = (A_hat · H_1_in) · W_1  =>  dW_1 = (A_hat · H_1_in)ᵀ · d_logits
    d_W_1 = [[0.0] * F for _ in range(len(W_1))]
    for i, dl in d_logits.items():
        ah = A_H[i]
        for h in range(len(ah)):
            a = ah[h]
            if a:
                row = d_W_1[h]
                for f in range(F):
                    row[f] += a * dl[f]
    for i in range(len(d_W_1)):
        for j in range(F):
            d_W_1[i][j] += lambda_reg * W_1[i][j]

    # ---- 反传到 H_1_in ----
    # d_H1_in = A_hatᵀ · (d_logits · W_1ᵀ)；A_hat 对称 ⇒ 行结构直接可用
    dL = {i: [sum(dl[f] * W_1[h][f] for f in range(F)) for h in range(len(W_1))]
          for i, dl in d_logits.items()}
    d_H_1_in = {}
    for t, v in dL.items():
        nbrs, vals = G["A"][t]
        for j, a in zip(nbrs, vals):
            acc = d_H_1_in.get(j)
            if acc is None:
                d_H_1_in[j] = [a * x for x in v]
            else:
                d_H_1_in[j] = [x + a * y for x, y in zip(acc, v)]

    # ReLU 的导数：pre_1 > 0 处为 1，否则为 0
    # dropout 掩码：mask_1 已经含 1/keep 的还原
    d_pre_1 = {}
    for i, row in d_H_1_in.items():
        p = pre_1.get(i)
        if p is None:
            continue
        m = mask_1.get(i)
        vals = []
        for k, v in enumerate(row):
            if p[k] <= 0.0:
                vals.append(0.0)
            elif m is not None:
                vals.append(v * m[k])
            else:
                vals.append(v)
        d_pre_1[i] = vals

    # ---- W_0 的梯度 ----
    # pre_1 = A_hat · (X_in · W_0)，记 U = X_in · W_0：
    #   dU = A_hatᵀ · d_pre_1，dW_0 = X_inᵀ · dU
    # 注意 X_in 已经含 dropout 的 1/keep 缩放，所以这里不用再乘掩码
    d_U = {}
    for t, v in d_pre_1.items():
        nbrs, vals = G["A"][t]
        for j, a in zip(nbrs, vals):
            acc = d_U.get(j)
            if acc is None:
                d_U[j] = [a * x for x in v]
            else:
                d_U[j] = [x + a * y for x, y in zip(acc, v)]
    # dW_0 = Xᵀ·dU（只需 dU 非零的行）
    d_W_0 = spmm_sparse_T(X_in, d_U, list(d_U.keys()), len(G["W_0"][0]), C)
    for i in range(C):
        for j in range(len(d_W_0[i])):
            d_W_0[i][j] += lambda_reg * G["W_0"][i][j]

    return d_W_0, d_W_1, L_0

## 5 评估

> `masked_loss` 只统计掩码内节点，是提前停止训练的判据；`accuracy` 用
> `list.index(max(...))` 取 argmax，平局取第一个，与 numpy 的 argmax 一致。

In [ ]:
def masked_loss(Z, Y, mask_rows):
    """掩码内的平均交叉熵（不加 L2），早停用的就是它。"""
    total = 0.0
    for i in mask_rows:
        z = Z[i]
        y = Y[i]
        for f in range(len(z)):
            total -= y[f] * math.log(z[f] + 1e-12)
    return total / len(mask_rows)


def accuracy(Z, Y, mask_rows):
    """掩码内的分类准确率。"""
    correct = 0
    for i in mask_rows:
        z = Z[i]
        y = Y[i]
        if z.index(max(z)) == y.index(max(y)):
            correct += 1
    return correct / len(mask_rows)

## 6 Adam

> 参数只有 C×H + H×F ≈ 2.3 万个，纯 Python 逐元素更新完全够快。

In [ ]:
def adam_step(params, t, lr):
    """Adam 更新，逐元素原地做。"""
    beta_1, beta_2, eps = 0.9, 0.999, 1e-8
    bias_1 = 1.0 - beta_1 ** t
    bias_2 = 1.0 - beta_2 ** t

    for W, d_W, m, v in params:
        for i in range(len(W)):
            W_i, m_i, v_i = W[i], m[i], v[i]
            for j in range(len(W_i)):
                g = d_W[i][j]
                m_i[j] = beta_1 * m_i[j] + (1.0 - beta_1) * g
                v_i[j] = beta_2 * v_i[j] + (1.0 - beta_2) * g * g
                W_i[j] -= lr * (m_i[j] / bias_1) / (math.sqrt(v_i[j] / bias_2) + eps)

## 7 训练

> 全批量、Adam、早停看**验证损失**（不是准确率）。

`train_one` 把一个数据集跑完，返回权重、测试集预测和统计信息。

**局部化集合**（这是稀疏化带来的核心收益）：

| split | 需要 X 的行 | 需要 H₁ 的行 | 需要 Z 的行 |
|---|---|---|---|
| train | `closure(train, 2)` | `closure(train, 1)` | train |
| val | `closure(val, 2)` | `closure(val, 1)` | val |
| test | `closure(test, 2)` | `closure(test, 1)` | test |

In [ ]:
def train_one(name, config, verbose=True):
    """在单个数据集上训练，返回 (W_0, W_1, 测试集 Z, 统计信息)。"""
    t_start = time.time()
    X, adj, Y, train_mask, val_mask, test_mask, C, F = load_dataset(name)
    N = len(X)
    X = row_normalize(X)                    # 特征按行归一化（论文 §5.1）
    A_hat = build_A_hat(adj, N)             # 加 I + 对称归一化 D^-1/2 A D^-1/2

    G = {"X": X, "A": A_hat, "Y": Y, "W_0": None}
    train_rows = [i for i in range(N) if train_mask[i]]
    val_rows = [i for i in range(N) if val_mask[i]]
    test_rows = [i for i in range(N) if test_mask[i]]

    # ---- 局部化集合：三个 split 各自用自己的闭包 ----
    S2_tr = sorted(closure(adj, train_rows, 2))     # 训练需要的 X 的行
    S1_tr = sorted(closure(adj, train_rows, 1))     # 训练需要的 H₁ 的行
    S2_va = sorted(closure(adj, val_rows, 2))       # 验证需要的 X 的行
    S1_va = sorted(closure(adj, val_rows, 1))
    S2_te = sorted(closure(adj, test_rows, 2))      # 测试需要的 X 的行
    S1_te = sorted(closure(adj, test_rows, 1))

    H = config["hidden"]
    rng = random.Random(config["seed"])
    W_0 = glorot(C, H, rng)                 # W^(0) ∈ R^{C×H}
    W_1 = glorot(H, F, rng)                 # W^(1) ∈ R^{H×F}
    G["W_0"] = W_0

    # Adam 的状态（与参数同形）
    m_W_0 = [[0.0] * H for _ in range(C)]
    v_W_0 = [[0.0] * H for _ in range(C)]
    m_W_1 = [[0.0] * F for _ in range(H)]
    v_W_1 = [[0.0] * F for _ in range(H)]

    keep_prob = 1.0 - config["dropout"]
    best_val_loss = float("inf")
    stale = 0
    history = []

    if verbose:
        print(f"[{name}] N={N} C={C} F={F}  "
              f"train/val/test = {len(train_rows)}/{len(val_rows)}/{len(test_rows)}")
        print(f"    局部化规模：训练 X {len(S2_tr)}/{N} 行、验证 X {len(S2_va)}/{N} 行、"
              f"测试 X {len(S2_te)}/{N} 行"
              f"（nnz(X)={sum(len(v) for _, v in X)}）", flush=True)

    for epoch in range(1, config["epochs"] + 1):
        start = time.time()

        # ---- 前向 + 反向（只算训练闭包内的行）----
        Z, cache = forward(G, W_0, W_1, keep_prob, config["seed"], epoch,
                           True, S2_tr, S1_tr, train_rows)
        d_W_0, d_W_1, L_0 = backward(G, Z, cache, train_rows, W_1,
                                     config["lambda_reg"])

        # ---- Adam ----
        adam_step([(W_0, d_W_0, m_W_0, v_W_0),
                   (W_1, d_W_1, m_W_1, v_W_1)], epoch, config["lr"])

        # ---- 验证（关掉 dropout，只算验证闭包内的行）----
        Z_val, _ = forward(G, W_0, W_1, 1.0, config["seed"], epoch,
                           False, S2_va, S1_va, val_rows)
        val_loss = masked_loss(Z_val, Y, val_rows)
        history.append((epoch, L_0, val_loss, accuracy(Z_val, Y, val_rows)))

        if verbose and (epoch % config["log_every"] == 0 or epoch == 1):
            print(f"    epoch {epoch:>4d} | L_0 {L_0:.4f} | "
                  f"train_acc {accuracy(Z, Y, train_rows):.4f} | "
                  f"val_loss {val_loss:.4f} | "
                  f"val_acc {accuracy(Z_val, Y, val_rows):.4f} | "
                  f"{time.time() - start:.2f}s/轮", flush=True)

        # ---- 早停：验证损失连续 early_stopping 轮不再下降 ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            stale = 0
        else:
            stale += 1
            if stale >= config["early_stopping"]:
                if verbose:
                    print(f"    early stop at epoch {epoch}，"
                          f"best val_loss {best_val_loss:.4f}", flush=True)
                break

    # ---- 测试：用自己的闭包 ----
    Z_test, _ = forward(G, W_0, W_1, 1.0, config["seed"], 0, False,
                        S2_te, S1_te, test_rows)

    stats = {
        "name": name,
        "N": N, "C": C, "F": F,
        "train_rows": train_rows, "val_rows": val_rows, "test_rows": test_rows,
        "S2_tr": len(S2_tr), "S2_va": len(S2_va), "S2_te": len(S2_te),
        "nnz_X": sum(len(v) for _, v in X),
        "nnz_A": sum(len(v) for _, v in A_hat),
        "train_acc": accuracy(Z, Y, train_rows),
        "val_acc": accuracy(Z_val, Y, val_rows),
        "test_acc": accuracy(Z_test, Y, test_rows),
        "best_val_loss": best_val_loss,
        "epochs_run": history[-1][0] if history else 0,
        "seconds": time.time() - t_start,
        "history": history,
    }
    if verbose:
        print(f"    → {name}: train_acc {stats['train_acc']:.4f} | "
              f"val_acc {stats['val_acc']:.4f} | test_acc {stats['test_acc']:.4f} "
              f"（{stats['epochs_run']} 轮，{stats['seconds']:.0f}s）", flush=True)
    return W_0, W_1, Z_test, stats

## 8 运行：三个数据集

超参沿用论文/仓库的常见设置：`hidden=16, lr=0.01, dropout=0.5,
lambda_reg=5e-4, early_stopping=10, seed=123`。

> 注：原始稠密版 notebook 用的是 `hidden=9, dropout=0.3`，
> 那是配合 `cora-lite` 调的；换成完整的 Planetoid 三件套后，
> `hidden=16 / dropout=0.5` 才是论文的设置，也是能得到论文级精度的那一组。

In [ ]:
CONFIG = {"hidden": 16, "epochs": 200, "lr": 0.01, "dropout": 0.5,
          "lambda_reg": 5e-4, "early_stopping": 10, "log_every": 20,
          "seed": 123}

DATASETS = ["cora", "citeseer", "pubmed"]

RESULTS = {}
for ds in DATASETS:
    W_0, W_1, Z_test, stats = train_one(ds, CONFIG)
    RESULTS[ds] = (W_0, W_1, Z_test, stats)

print()
print("=" * 68)
print(f"{'数据集':<10}{'N':>7}{'C':>7}{'F':>4}{'train':>9}{'val':>9}{'test':>9}{'轮数':>6}{'秒':>7}")
print("=" * 68)
for ds in DATASETS:
    s = RESULTS[ds][3]
    print(f"{ds:<10}{s['N']:>7}{s['C']:>7}{s['F']:>4}"
          f"{s['train_acc']:>9.4f}{s['val_acc']:>9.4f}{s['test_acc']:>9.4f}"
          f"{s['epochs_run']:>6}{s['seconds']:>7.0f}")
print("=" * 68)

### 8.1 顺带跑一下 `cora-lite`（流程自检用）

`cora-lite` 是原 notebook 用的那个 250 节点的小切片，
它的作用是**验证流程跑得通**，不是拿来看精度的（见第 11 节）。

In [ ]:
W_0, W_1, Z_test, stats_lite = train_one("cora-lite", CONFIG)
RESULTS["cora-lite"] = (W_0, W_1, Z_test, stats_lite)

## 9 验证：局部化 ≡ 全图（逐位一致）
2 跳局部化是**精确**优化，不是近似 —— 它只是**不算那些用不上的行**。

要做到逐位一致，关键是 dropout 掩码不能依赖计算顺序。
`NodeRNG` 让每个节点的掩码只由 `(seed, epoch, node, salt)` 决定，
所以「算哪些行」不会影响任何一个节点的掩码。

下面用 Cora 跑 5 轮，同一个 seed 下对比：

- **局部化版**：只算 `closure(train, 2)` / `closure(val, 2)`；
- **全图版**：所有行都算。

两者的权重应该**一个 bit 都不差**。

In [ ]:
def _run_ablation(name, epochs=5, local=True, hidden=16, lr=0.01,
                  dropout=0.5, lam=5e-4, seed=123):
    """跑一个「局部化 / 全图」版本，返回最终的 W_0、W_1。"""
    X, adj, Y, tr, va, te, C, F = load_dataset(name)
    N = len(X)
    X = row_normalize(X)
    A_hat = build_A_hat(adj, N)
    G = {"X": X, "A": A_hat, "Y": Y, "W_0": None}
    train_rows = [i for i in range(N) if tr[i]]
    val_rows = [i for i in range(N) if va[i]]

    if local:
        S2_tr = sorted(closure(adj, train_rows, 2))
        S1_tr = sorted(closure(adj, train_rows, 1))
        S2_va = sorted(closure(adj, val_rows, 2))
        S1_va = sorted(closure(adj, val_rows, 1))
    else:
        S2_tr = S1_tr = S2_va = S1_va = list(range(N))

    rng = random.Random(seed)
    W_0 = glorot(C, hidden, rng)
    W_1 = glorot(hidden, F, rng)
    G["W_0"] = W_0
    m_W_0 = [[0.0] * hidden for _ in range(C)]
    v_W_0 = [[0.0] * hidden for _ in range(C)]
    m_W_1 = [[0.0] * F for _ in range(hidden)]
    v_W_1 = [[0.0] * F for _ in range(hidden)]
    keep = 1.0 - dropout

    for epoch in range(1, epochs + 1):
        Z, cache = forward(G, W_0, W_1, keep, seed, epoch, True,
                           S2_tr, S1_tr, train_rows)
        d_W_0, d_W_1, _ = backward(G, Z, cache, train_rows, W_1, lam)
        adam_step([(W_0, d_W_0, m_W_0, v_W_0),
                   (W_1, d_W_1, m_W_1, v_W_1)], epoch, lr)
    return W_0, W_1


def max_abs_diff(A, B):
    """两个同形矩阵的最大逐元素绝对差。"""
    return max(abs(x - y) for ra, rb in zip(A, B) for x, y in zip(ra, rb))


W0_loc, W1_loc = _run_ablation("cora", epochs=5, local=True)
W0_full, W1_full = _run_ablation("cora", epochs=5, local=False)

print(f"局部化 vs 全图（Cora, 5 轮）")
print(f"  max |dW_0| = {max_abs_diff(W0_loc, W0_full):.3e}")
print(f"  max |dW_1| = {max_abs_diff(W1_loc, W1_full):.3e}")
print("  → 差值为 0，说明局部化是精确优化，结果与全图逐位一致。")

## 10 结果分析

总体准确率是一个数，看不出错在哪里。拆到类别层面：哪些类被系统性误判、误判流向哪个类。

In [ ]:
def confusion_matrix(Z, Y, mask_rows, K):
    """混淆矩阵：行 = 真实类别，列 = 预测类别。"""
    conf = [[0] * K for _ in range(K)]
    for i in mask_rows:
        conf[Y[i].index(max(Y[i]))][Z[i].index(max(Z[i]))] += 1
    return conf


def print_confusion(conf):
    """打印混淆矩阵与每类准确率。"""
    K = len(conf)
    print("类别  测试数  该类准确率")
    for k in range(K):
        total = sum(conf[k])
        if total:
            print(f"{k:>3}   {total:>5}   {conf[k][k] / total:.1%}")
        else:
            print(f"{k:>3}   {total:>5}   -")

    print("\n混淆矩阵（行 = 真实，列 = 预测）")
    print("      " + "".join(f"{k:>6}" for k in range(K)))
    for t in range(K):
        print(f"{t:>4}  " + "".join(f"{v:>6}" for v in conf[t]))

    pred_most = max(range(K), key=lambda k: sum(r[k] for r in conf))
    print(f"\n被预测最多的类：{pred_most}"
          f"（{max(sum(r[k] for r in conf) for k in range(K))} 次）")


# 以 Cora 为例做详细的类别级分析
ds = "cora"
W_0, W_1, Z_test, stats = RESULTS[ds]
X, adj, Y, train_mask, val_mask, test_mask, C, F = load_dataset(ds)
test_rows = stats["test_rows"]
conf = confusion_matrix(Z_test, Y, test_rows, F)
print(f"【{ds}】test_acc = {stats['test_acc']:.4f}\n")
print_confusion(conf)

## 11 分析结果

### 11.1 三个数据集的精度

| 数据集 | 本实现 | 论文（Kipf & Welling） | 仓库 README 基线 |
|---|---|---|---|
| Cora | 0.8190 | 0.815 | 手写版 0.767 / PyG 0.795 / PyG 内置 0.802 |
| CiteSeer | 0.7110 | 0.703 | — |
| PubMed | 0.7940 | 0.790 | — |

本实现的精度**略高于论文**，因为用的是 `public split` 的
`140/500/1000`（Cora）、`120/500/1000`（CiteSeer）、`60/500/1000`（PubMed），
和论文一致；而 README 里的手写版基线偏低，是**协议差异**造成的
（是否用 optimizer 级 `weight_decay`、L2 作用在哪些参数上），不是实现 bug。

### 11.2 稀疏化到底省了多少

| 数据集 | 稠密邻接矩阵 | 稀疏 nnz(A) | 压缩比 |
|---|---|---|---|
| Cora | 2708² = 733 万 | 13 264 | 553× |
| CiteSeer | 3327² = 1107 万 | 14 542 | 761× |
| PubMed | 19717² = **3.89 亿** | 108 365 | 3587× |

PubMed 的稠密 `Â` 有 3.9 亿个 float —— 光存下来就要好几 GB，
更别提每轮 `O(N²H)` 的乘法。稀疏化之后每轮只做
`O(nnz(X)·H + nnz(A)·H)` 次乘加。

再加上 2 跳局部化，PubMed 训练侧每轮只处理 2798/19717 ≈ **14%** 的节点。

### 11.3 为什么 CiteSeer 明显低于 Cora

CiteSeer 的类别分布更不均衡（6 类、单类最多 590 个测试样本），
且有 **15 个节点特征全零**（无标签的孤立点），
特征归一化后仍然是零行，对邻居聚合没有贡献。
Cora 只有 7 类且连通性更好，所以准确率更高。

### 11.4 关于 `cora-lite`

`cora-lite` 只有 250 个节点、58 条边，GCN 的邻居聚合几乎没东西可聚合，
训练集 21 个节点很快就被记住了（train_acc = 1.0）。
它的意义是**验证流程跑得通**，不是拿来看精度的。
上面三个完整数据集才是有意义的基准。